# Pista B — motor de decisión en simulación (B1–B5)

**19 de septiembre de 2026.** Corre la pista B de `docs/experimentos_productos.md` §4 sobre el
dataset real: validar el simulador de caja (B1), comparar líneas base (B2), el MPC (B3), una
política aprendida por iteración Q ajustada (B4) y barrer el precio sombra de la rotura (B5).

Este cuaderno **importa `xray`, no define el motor**: el simulador es `xray.projection`
(`SimConfig`, `History`, `simulate`, `advance`, `backtest`) y las políticas son `xray.policies`
(`advisor_rules`, `miller_orr`, `adl_refinance`, `make_mpc_policy`, `evaluate_policy`,
`perfect_foresight`). Lo único que se escribe aquí es el FQI de B4, que es el experimento.

Salidas en `artifacts/experiments/`: `B_extras.parquet` (caché del estado de productos),
`B1_calibration.csv/png`, `B1_validation.csv`, `B2_baselines.csv`, `B3_mpc.csv`,
`B3_tradeoff.png`, `B4_rl.csv`, `B4_importance.png`, `B5_frontier.csv/png` y `B_summary.json`.

**Advertencias que valen para todo el cuaderno** (se repiten donde tocan y van en el JSON final):

1. Los compromisos que escribe `advance` **no llevan plazo**: la cuota de un préstamo se sigue
   cobrando más allá de su vencimiento en rollouts largos. A 6–12 meses con préstamos a 36 no
   cambia nada; a 24 meses sí. Y el interés del préstamo se compromete **plano**, al tipo del
   primer mes, porque un float no lleva cuadro de amortización (el error va del lado caro).
2. La banda 10–90 % del saldo cubre ~72 %, no 80 %: los sorteos son i.i.d. y la banda sale
   estrecha. Se enseña la cobertura medida, no la nominal.
3. La `P(rotura)` cruda es **muy optimista al revés**: sobreestima. Ordena bien (AUC) y calibra
   mal; el número que se enseña es el calibrado con la isotónica de entrenamiento.
4. El coste en euros de una tabla de políticas está dominado por **una sola empresa** del
   hold-out (`COMP_1185`, con 2.700 M€ de cargos mensuales medianos). Todas las tablas llevan al
   lado el coste normalizado por la mediana de cargos de cada empresa, que es el que compara.

In [ ]:
import json
import time
import warnings
from collections import Counter
from dataclasses import replace

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

from xray import policies as pl
from xray import projection as pj
from xray import rules
from xray.data import artifacts_dir, load

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

T0 = time.time()
SEED = 0
OUT = artifacts_dir() / "experiments"
OUT.mkdir(parents=True, exist_ok=True)

CFG = pj.SimConfig(n_paths=500, horizon=6, seed=SEED)
BACKTEST_MONTHS = [str(m) for m in pd.period_range("2025-09", "2026-02", freq="M")]
TRAIN_UNTIL = "2025-08"
START_MONTH = "2025-08"       # el mes en el que arrancan todos los bucles cerrados
LAST_MONTH = "2026-08"        # exigido a la empresa para tener 12 meses reales por delante
MIN_HISTORY = 6               # meses de historia mínimos para entrar en el hold-out
CLOSED_LOOP_MONTHS = 12       # B2/B3/B4
ORACLE_MONTHS = 6             # el oráculo solo es comparable a su propio horizonte
MPC_PATHS = 200               # caminos del MPC dentro del bucle (la ficha en pantalla usa 500)
MPC_K = (0.1, 0.5, 2.0)       # λ = k × mediana de cargos
FRONTIER_K = (0.1, 0.25, 0.5, 1.0, 2.0, 4.0)
FRONTIER_N = 100
FRONTIER_PATHS = 100
SHIFT = {"inflow_scale": 0.8, "rate_shock": 0.02, "dip_scale": 1.3}
print(f"salidas en {OUT}")

## Carga

`features.parquet` es la tabla del seam `features(company_id, month)`; `load()` trae las tablas
crudas del reto desde la caché parquet. `projection.company_extras` añade lo que las features no
llevan y el simulador necesita para saber qué productos son elegibles (límite y dispuesto de la
póliza, cartera emitida viva, saldo vivo, cuota, tipo y plazo del préstamo). Se cachea en
`B_extras.parquet` porque tarda un par de segundos y lo lee también el cuaderno 05.

`FlowPool.fit` ajusta el pool de tripletes normalizados con el que el simulador sortea el futuro
de las empresas con historia corta. **Se pasa siempre**: sin él, una empresa de tres meses sale
con una banda de ancho cero y el AUC del backtest baja (se mide más abajo).

In [ ]:
features = pd.read_parquet(artifacts_dir() / "features.parquet")
tables = load()
months = sorted(features["month"].astype(str).unique())

extras_path = OUT / "B_extras.parquet"
if extras_path.exists():
    extras = pd.read_parquet(extras_path)
else:
    extras = pj.company_extras(
        tables["transactions"], tables["debt_products"], tables["banking_products"],
        tables["debt_schedule_config"], tables["invoices"], months, features=features,
    )
    extras.to_parquet(extras_path, index=False)

pool = pj.FlowPool.fit(features)
hists_all = pj.histories(features, extras, CFG)

print(f"features {features.shape} · {features['company_id'].nunique()} empresas · "
      f"{len(months)} meses ({months[0]} … {months[-1]})")
print(f"extras {extras.shape} · pool {pool.triplets.shape} · {len(hists_all)} historias")
print(f"empresas con tipo de contrato conocido: {extras['loan_rate'].notna().sum() // len(months)}")

## B1 — El simulador, validado por backtest

`projection.backtest` simula sin acción todas las filas de entrenamiento (`month <= 2025-08`) y
las de test (2025-09 … 2026-02), compara `p_breach_raw` con el descubierto realizado a 1 y 6
meses, ajusta la isotónica **solo en entrenamiento** (y solo sobre filas con saldo mínimo ≥ 0,
para no medir lo que ya está roto) y mide la cobertura de la banda 10–90 % del saldo.

Se corre dos veces, con pool y sin pool, para dejar medido lo que aporta el shrinkage.

In [ ]:
t0 = time.perf_counter()
bt_cfg = replace(CFG, n_paths=pj.BACKTEST_PATHS)
metrics_pool, bt_table = pj.backtest(features, extras, bt_cfg, BACKTEST_MONTHS,
                                     train_until=TRAIN_UNTIL, pool=pool)
metrics_nopool, _ = pj.backtest(features, extras, bt_cfg, BACKTEST_MONTHS,
                                train_until=TRAIN_UNTIL, pool=None)
print(f"backtest (con y sin pool) en {time.perf_counter() - t0:.1f} s")

compare = pd.DataFrame({
    "con pool": {k: v for k, v in metrics_pool.items() if k != "calibration"},
    "sin pool": {k: v for k, v in metrics_nopool.items() if k != "calibration"},
})
print(compare.round(4).to_string())

Lo que dicen los números, en el orden en que importan:

- **Ordena**: AUC 0,79 a un mes y 0,69 a seis entre las empresas que todavía no están rotas. El
  criterio de B1 pedía ≥ 0,70 a seis meses y se queda dos puntos corto.
- **No calibra sola**: el decil más alto predice 0,94 de probabilidad cruda frente a 0,23
  realizada. La isotónica lo arregla en nivel (0,19 frente a 0,23) sin tocar el orden (el AUC
  calibrado es el mismo salvo empates que la isotónica aplana).
- **La banda es estrecha**: cubre 72–74 % en vez del 80 % nominal, fuera del 80 ± 8 % que pedía
  el criterio por poco. La causa es de diseño: los tripletes se sortean i.i.d., sin persistencia
  de régimen, así que la dispersión del saldo acumulado se queda corta.
- **El pool vale**: con shrinkage el AUC a un mes sube de 0,758 a 0,795 y la cobertura de 65 % a
  72 %. Es el argumento para pasarlo siempre.

In [ ]:
calibration = pd.DataFrame(metrics_pool["calibration"])
calibration.to_csv(OUT / "B1_calibration.csv", index=False)
print(calibration.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
ax.plot(calibration["p_raw"], calibration["realized"], "o-", label="cruda")
ax.plot(calibration["p_cal"], calibration["realized"], "s-", label="calibrada")
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="identidad")
ax.set_xlabel("P(rotura a 6 meses) predicha")
ax.set_ylabel("frecuencia realizada")
ax.set_title("Calibración por decil (test 2025-09…2026-02)")
ax.legend(fontsize=8)

ax = axes[1]
horizons = [1, 3, 6]
cov = [metrics_pool[f"coverage_p10_p90_h{h}"] for h in horizons]
cov_no = [metrics_nopool[f"coverage_p10_p90_h{h}"] for h in horizons]
x = np.arange(len(horizons))
ax.bar(x - 0.2, cov, 0.4, label="con pool")
ax.bar(x + 0.2, cov_no, 0.4, label="sin pool")
ax.axhline(0.8, color="k", ls="--", lw=0.8)
ax.axhspan(0.72, 0.88, color="grey", alpha=0.15)
ax.set_xticks(x, [f"h={h}" for h in horizons])
ax.set_ylabel("cobertura de la banda 10–90 %")
ax.set_title("Cobertura del saldo frente al 80 ± 8 % pedido")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "B1_calibration.png", dpi=140)
plt.close(fig)

### B1.ii — ¿aporta el simulador algo sobre el score de reglas?

El score de `xray.rules` ya ordena el riesgo con la información del mes. La pregunta honesta es
si el simulador añade algo: se mide el AUC de `−score` contra la misma etiqueta y sobre las
**mismas filas** (test, saldo mínimo ≥ 0, con `breach6` observable).

In [ ]:
scored = rules.run(features)
score_col = scored[["company_id", "month", "score"]].copy()
score_col["month"] = score_col["month"].astype(str)
evaluated = bt_table[(bt_table["split"] == "test") & (bt_table["min_balance_eur"] >= 0)].merge(
    score_col, on=["company_id", "month"], how="left")
ok = evaluated["breach6"].notna() & evaluated["score"].notna()
auc_rows = {
    "score de reglas (-score)": roc_auc_score(evaluated.loc[ok, "breach6"],
                                              -evaluated.loc[ok, "score"]),
    "simulador (cruda)": roc_auc_score(evaluated.loc[ok, "breach6"],
                                       evaluated.loc[ok, "p_breach_raw"]),
    "simulador (calibrada)": roc_auc_score(evaluated.loc[ok, "breach6"],
                                           evaluated.loc[ok, "p_breach_cal"]),
}
print(f"filas comparadas: {int(ok.sum())} de {len(evaluated)}")
for name, value in auc_rows.items():
    print(f"  {name:<26} AUC {value:.4f}")

**Empatan.** 0,6987 el score de reglas y 0,6981 el simulador sobre las mismas 4.963 filas. El
simulador **no** se justifica como clasificador: se justifica porque responde a una pregunta que
el score no puede contestar —*¿y si abro una póliza de 200 K€?*— y porque devuelve euros y una
banda de saldo, no un número del 0 al 100. Es exactamente lo que hay que decir en el pitch.

### B1.iii — la mecánica del producto frente a los ATT observados

Tercera validación del encargo: coger los eventos de adopción **limpios** de A1 (≥ 6 meses de
historia antes y después, con importe conocido) y preguntarle al simulador cuánto habría movido
la probabilidad de rotura *ese* producto por *ese* importe. Si la mecánica del simulador
reproduce lo que se observó en A2, los dos números tienen que ser compatibles.

Se simula **en el mes anterior al evento**, no en el del evento. Una `History` es el estado al
**cierre** de su mes, así que la del mes del evento ya lleva dentro el ingreso de la disposición
y la primera cuota del préstamo: comparar `NONE` contra el producto ahí sería comparar la
empresa con el producto ya puesto contra ella misma con el producto puesto dos veces. La
referencia correcta es `t − 1`, que es la empresa antes de adoptarlo.

- `disposicion/description` y `line/line_tx` → `line_draw(min(importe, disponible))`, y
  `line_open(importe)` cuando la empresa no tiene póliza que disponer.
- `loan/installment` → el préstamo cuyo **principal reproduce la cuota observada** al plazo del
  simulador (`SimConfig.loan_term_months = 36`), no `24 × cuota`: invirtiendo la anualidad,
  `A = cuota · (1 − (1+i)^−36) / i` con `i = fair_loan_rate(A)/12`. Como el tipo depende del
  tramo y el tramo del importe, se arranca en el tramo de ≤ 250 K€ y se reasigna una vez (en
  estos 78 eventos converge siempre). Con `24 × cuota` el principal salía a 0,70× del que hace
  falta y el Δ servicio de deuda simulado no cuadraba con la cuota observada; ahora sí, por
  construcción, y eso es lo que convierte la fila en una comprobación de verdad.

La diferencia se mide con los **mismos sorteos** (números aleatorios comunes) y se reporta cruda
y calibrada con la isotónica de entrenamiento del backtest, que es la escala en la que el ATT de
A2 está medido (una frecuencia observada).

In [ ]:
train_rows = bt_table[(bt_table["split"] == "train") & (bt_table["min_balance_eur"] >= 0)
                      & bt_table["breach6"].notna()]
calibrator = pj.calibrate(train_rows["p_breach_raw"], train_rows["breach6"])

def loan_principal(installment, cfg=CFG):
    """Principal que produce `installment` a `cfg.loan_term_months` al tipo de su propio tramo.

    Invierte la anualidad, `A = cuota · (1 − (1+i)^−n) / i`. El tipo depende del tramo y el tramo
    del importe, así que se arranca en el tramo más barato (≤ 250 K€) y se reasigna una vez: basta
    porque los tramos están ordenados y la anualidad es monótona en el tipo. Si en un caso límite
    no coincidiera, el tipo que usa `simulate` es el del importe final, no este.
    """
    rate = float(cfg.loan_rates[0][1])
    principal = float("nan")
    for _ in range(2):
        i = rate / 12.0
        principal = float(installment) * (1 - (1 + i) ** -cfg.loan_term_months) / i
        bucket = pl.fair_loan_rate(principal, cfg)
        if bucket == rate:
            break
        rate = bucket
    return principal


events = pd.read_csv(OUT / "A1_events.csv")
VALIDATION_PAIRS = [("disposicion", "description"), ("line", "line_tx"), ("loan", "installment")]
validation_rows = []
for product, source in VALIDATION_PAIRS:
    clean = events[(events["product"] == product) & (events["source"] == source)
                   & (events["pre_months"] >= 6) & (events["post_months"] >= 6)
                   & (events["amount_eur"] > 0)]
    clean = clean.sort_values("month").drop_duplicates(["company_id"])
    raw, cal, service, kinds, missing = [], [], [], [], 0
    for company, month, amount in clean[["company_id", "month", "amount_eur"]].itertuples(
            index=False, name=None):
        # El mes ANTERIOR al evento: la `History` de `month` ya lleva el producto dentro.
        hist = hists_all.get((company, str(pd.Period(str(month), freq="M") - 1)))
        if hist is None:
            missing += 1
            continue
        if product == "loan":
            action = pj.Action("loan", amount=loan_principal(float(amount)))
        else:
            room = float(hist.line_limit) - float(hist.line_drawn)
            action = (pj.Action("line_draw", amount=min(float(amount), room)) if room > 0
                      else pj.Action("line_open", amount=float(amount)))
        base = pj.simulate(hist, pj.NONE, CFG, rng=np.random.default_rng(SEED), pool=pool)
        try:
            alt = pj.simulate(hist, action, CFG, rng=np.random.default_rng(SEED), pool=pool)
        except ValueError:
            continue  # el producto no es elegible en ese mes concreto
        kinds.append(action.kind)
        raw.append(alt.breach_prob() - base.breach_prob())
        cal.append(float(calibrator.predict([alt.breach_prob()])[0]
                         - calibrator.predict([base.breach_prob()])[0]))
        service.append(float(alt.debt_service[:, -1].mean() - base.debt_service[:, -1].mean()))
    validation_rows.append({
        "product": product, "source": source, "n_events": len(clean), "n_simulated": len(raw),
        "n_sin_historia_t_1": missing,
        "actions": ", ".join(f"{k}×{v}" for k, v in Counter(kinds).most_common()),
        "median_amount_eur": float(clean["amount_eur"].median()) if len(clean) else np.nan,
        "sim_d_breach6_raw": float(np.mean(raw)) if raw else np.nan,
        "sim_d_breach6_cal": float(np.mean(cal)) if cal else np.nan,
        "sim_d_debt_service_m": float(np.mean(service)) if service else np.nan,
        "sim_d_debt_service_m_median": float(np.median(service)) if service else np.nan,
    })

att = pd.read_csv(OUT / "A2_att.csv")
att6 = att[(att["outcome"] == "breach6") & (att["h"] == 6) & (att["estimator"] == "matched")]
validation = pd.DataFrame(validation_rows).merge(
    att6[["product", "source", "n_events", "att", "se", "ci_lo", "ci_hi", "pretrend_att"]].rename(
        columns={"n_events": "a2_n_events", "att": "a2_att", "se": "a2_se",
                 "ci_lo": "a2_ci_lo", "ci_hi": "a2_ci_hi", "pretrend_att": "a2_pretrend"}),
    on=["product", "source"], how="left")
validation["inside_a2_ci"] = ((validation["sim_d_breach6_cal"] >= validation["a2_ci_lo"])
                              & (validation["sim_d_breach6_cal"] <= validation["a2_ci_hi"]))
validation.to_csv(OUT / "B1_validation.csv", index=False)
print(validation.drop(columns=["source", "a2_se"]).round(4).to_string(index=False))
loan_row = validation[validation["product"] == "loan"].iloc[0]
print(f"\ncontrol del dimensionado del préstamo: cuota observada (mediana) "
      f"{loan_row['median_amount_eur']:,.2f} € · Δ servicio de deuda simulado (mediana) "
      f"{loan_row['sim_d_debt_service_m_median']:,.2f} €")

Los 98 eventos tienen historia en `t − 1`, así que no se pierde ninguno por mover la referencia
un mes atrás. Dos de tres caen dentro del intervalo de A2:

- **Disposición de póliza**: el simulador da −3,1 puntos de rotura calibrados; A2 estimó −6,9
  puntos [−23,8; +10,5] con n = 7. Compatible, y con la pre-tendencia de A2 en cero.
- **Línea (movimiento de cuenta)**: −7,4 puntos simulados frente a −9,0 [−12,9; −4,0] de A2.
  Compatible, y es el único ATT de A2 con intervalo que no cruza el cero.
- **Préstamo**: el simulador dice −3,1 puntos y A2 midió **+10,0** [−0,1; +21,4]. El simulado se
  queda fuera por abajo, y el signo contrario es lo esperable: A2 mide **quién pide un préstamo**
  (empresas que ya iban mal: es la selección que documenta §2.4), mientras que el simulador mide
  **qué hace el préstamo** manteniendo todo lo demás igual. Los dos números son correctos y
  responden a preguntas distintas; en la demo no se pueden intercambiar.

La otra mitad de la hipótesis de A2 —«baja la rotura **a costa del DSCR**»— también aparece, y
ahora cuadra: el préstamo sube el servicio de deuda en **5.865 €/mes en la mediana, que es
exactamente la cuota observada** (54.287 € de media, que es la cola de importes). Esa igualdad no
es un hallazgo, es el control de que el dimensionado del principal está bien hecho: con el
`24 × cuota` del encargo salía 4.128 €, un 30 % corto. Los productos de póliza no tocan el
servicio de deuda, como tiene que ser.

La diferencia cruda (sin calibrar) es mucho mayor —−0,23, −0,42 y −0,25— porque la probabilidad
cruda vive en una escala inflada. Por eso la comparación con A2 se hace en la escala calibrada.

## Partición: 20 % de los grupos, retenido

El hold-out es el primer fold de `GroupKFold(5)` sobre `group_id` (no sobre empresa: hay 248
grupos y las empresas de un grupo comparten dueño). De ahí se queda con las historias del mes
**2025-08** que tengan ≥ 6 meses de historia y cuya empresa llegue a 2026-08 en la tabla de
features, o sea, 12 meses reales por delante.

**Cualifican 126 empresas, por debajo de las 150 del encargo.** La relajación prevista (exigir
solo 6 meses por delante y evaluar a 6 meses) sube a 130 y tampoco llega a 150, así que se
mantiene el criterio estricto y los 12 meses de bucle cerrado: el bucle cerrado **no consume
meses reales de futuro** —el mundo lo sortea el simulador—, de modo que el requisito de 12 meses
solo sirve para exigir que la empresa siga viva en el panel, y bajar el horizonte a cambio de
cuatro empresas sería un mal negocio. Queda anotado como desviación.

In [ ]:
company_ids = np.array(sorted(features["company_id"].unique()))
group_of = tables["companies"].set_index("company_id")["group_id"].reindex(company_ids)
train_idx, hold_idx = next(GroupKFold(n_splits=5).split(company_ids, groups=group_of.to_numpy()))
hold_ids, train_ids = set(company_ids[hold_idx]), set(company_ids[train_idx])
last_month = features.assign(m=features["month"].astype(str)).groupby("company_id")["m"].max()


def qualifying(ids, last_needed=LAST_MONTH):
    """Historias de `ids` en `START_MONTH` con historia suficiente y futuro observable."""
    keys = [(c, START_MONTH) for c in sorted(ids) if (c, START_MONTH) in hists_all]
    return [hists_all[k] for k in keys
            if len(hists_all[k].inflows) >= MIN_HISTORY and last_month.get(k[0], "") >= last_needed]


hold_hists = qualifying(hold_ids)
train_hists = qualifying(train_ids)
hold_med = np.array([float(np.median(h.outflows)) for h in hold_hists])


def refinance_eligible(hist, cfg=CFG):
    """Igual que la elegibilidad de `_plan`: tipo conocido, deuda viva, cuota, plazo y mejora."""
    rate = hist.loan_rate
    if rate is None or not np.isfinite(float(rate)):
        return False
    if (float(hist.loan_outstanding) <= 0 or float(hist.loan_installment) <= 0
            or int(hist.loan_remaining) < 1):
        return False
    return float(rate) > pl.fair_loan_rate(float(hist.loan_outstanding), cfg)


n_line_room = sum(1 for h in hold_hists if h.line_limit - h.line_drawn > 0)
n_refi = sum(1 for h in hold_hists if refinance_eligible(h))
n_receivables = sum(1 for h in hold_hists if h.receivables > 0)
print(f"grupos {group_of.nunique()} · hold-out {len(hold_ids)} empresas, cualifican "
      f"{len(hold_hists)} · train {len(train_ids)} empresas, cualifican {len(train_hists)}")
print(f"relajando a 6 meses de futuro cualificarían {len(qualifying(hold_ids, '2026-02'))}")
print(f"elegibilidad en el hold-out: póliza con disponible {n_line_room}, cartera "
      f"{n_receivables}, refinanciación {n_refi}")
print(f"mediana de cargos: p25 {np.percentile(hold_med, 25):,.0f} € · "
      f"mediana {np.median(hold_med):,.0f} € · p75 {np.percentile(hold_med, 75):,.0f} € · "
      f"máximo {hold_med.max():,.0f} € · {int((hold_med == 0).sum())} empresas con 0")

## B2 — Líneas base

Cuatro políticas del encargo, todas en bucle cerrado 12 meses sobre las mismas 126 historias y
con **las mismas semillas de mundo** (`evaluate_policy` usa `[seed, i, j]`), así que la
diferencia entre políticas no es ruido de sorteo.

`miller_orr` y `adl_refinance` **solo pueden actuar si la empresa tiene el producto**: la
primera necesita póliza con disponible (17 de 126) y la segunda un tipo de contrato conocido y
mejorable (**0 de 126**, porque `debt_schedule_config` solo cubre 87 contratos en todo el
dataset). Por eso `combine(adl_refinance, advisor_rules)` sale **idéntica** a `advisor_rules`:
no es un empate, es que la pata de ADL nunca dispara en este hold-out. Se reporta el N elegible
al lado de cada línea base para que el empate no se lea como un resultado.

Sobre el coste: se dan los euros (que es lo que paga la empresa) y el **coste relativo**
`coste / mediana de cargos mensuales`, en «meses de cargos». La media en euros está dominada por
una empresa —`COMP_1185` se lleva el 95 % del coste total del MPC— y el relativo es el único
número que compara una pyme de 20 K€ con una de 2.700 M€.

In [ ]:
UNIT = np.where(hold_med > 0, hold_med, np.nan)  # 5 empresas con mediana de cargos 0 → sin relativo


def summarise(name, result, seconds, n_eligible, months=CLOSED_LOOP_MONTHS):
    """Fila de tabla de una corrida de `evaluate_policy`, con coste absoluto y relativo."""
    cost = result["total_cost"].to_numpy(float)
    relative = cost / UNIT if len(cost) == len(UNIT) else cost / np.nan
    kinds = Counter(k for actions in result["actions"] for k in actions)
    total = sum(kinds.values())
    switches = sum(sum(1 for a, b in zip(acts, acts[1:]) if a != b) for acts in result["actions"])
    pairs = sum(max(0, len(acts) - 1) for acts in result["actions"])
    return {
        "policy": name,
        "n_companies": len(result),
        "n_eligible": n_eligible,
        "mean_cost_eur": float(np.mean(cost)),
        "median_cost_eur": float(np.median(cost)),
        "p90_cost_eur": float(np.percentile(cost, 90)),
        "mean_cost_rel": float(np.nanmean(relative)),
        "median_cost_rel": float(np.nanmedian(relative)),
        "breach_rate": float(result["breach"].mean()),
        "breach_months": int(result["n_breach_months"].sum()),
        "dscr_fail_months": int(result["dscr_fail_months"].sum()),
        "share_none": kinds.get("none", 0) / total if total else np.nan,
        "switch_rate": switches / pairs if pairs else np.nan,
        "s_per_company_month": seconds / (len(result) * months),
        "action_mix": dict(kinds.most_common()),
    }


baseline_specs = {
    "do_nothing": (pl.do_nothing, len(hold_hists)),
    "advisor_rules": (pl.advisor_rules, len(hold_hists)),
    "adl+advisor_rules": (pl.combine(pl.adl_refinance, pl.advisor_rules), n_refi),
    "miller_orr": (pl.miller_orr, n_line_room),
}
baseline_runs, baseline_rows = {}, []
for name, (policy, n_eligible) in baseline_specs.items():
    t0 = time.perf_counter()
    run = pl.evaluate_policy(policy, hold_hists, CFG, months=CLOSED_LOOP_MONTHS, seed=SEED,
                             pool=pool)
    baseline_runs[name] = run
    baseline_rows.append(summarise(name, run, time.perf_counter() - t0, n_eligible))

baselines = pd.DataFrame(baseline_rows)
baselines.assign(action_mix=baselines["action_mix"].astype(str)).to_csv(
    OUT / "B2_baselines.csv", index=False)
print(baselines.drop(columns="action_mix").round(4).to_string(index=False))
print()
for row in baseline_rows:
    print(f"  {row['policy']:>18}: {row['action_mix']}")

Las reglas del asesor hacen lo que se les pide: bajan la rotura del 38,9 % al 27,8 % y el coste
relativo **también baja**, de 0,618 a 0,580 meses de cargos, porque el descubierto al 18 % es más
caro que la póliza al 3,75 %. En euros crudos, en cambio, sube (26,6 K€ → 31,7 K€). La
contradicción entera es la empresa grande: el relativo es el número que hay que leer.

Miller–Orr apenas actúa (12 disposiciones en 1.512 empresa-mes) porque solo 17 empresas tienen
póliza con disponible, y con ese N no separa nada del «no hacer nada».

## B3 — MPC

El MPC enumera los candidatos elegibles, simula seis meses de cada uno con los **mismos
sorteos** y se queda con el que minimiza `E[coste] + λ·P(rotura) + μ·P(DSCR < 1,2)`. λ y μ van
en la escala de cada empresa: `λ = k × mediana de cargos` y `μ = λ/4`, con `k ∈ {0,1; 0,5; 2}`.
Tres puntos, no uno, porque **λ es una decisión de negocio y no un parámetro estimado**
(`docs/experimentos_productos.md` §6).

In [ ]:
def lam_for(k):
    return lambda hist: k * float(np.median(hist.outflows))


def mu_for(k):
    return lambda hist: 0.25 * k * float(np.median(hist.outflows))


mpc_runs, mpc_rows = {}, []
for k in MPC_K:
    policy = pl.make_mpc_policy(lam_for(k), mu_for(k), pool=pool, n_paths=MPC_PATHS)
    t0 = time.perf_counter()
    run = pl.evaluate_policy(policy, hold_hists, CFG, months=CLOSED_LOOP_MONTHS, seed=SEED,
                             pool=pool)
    mpc_runs[k] = run
    mpc_rows.append(summarise(f"mpc_k{k}", run, time.perf_counter() - t0, len(hold_hists)))

mpc_table = pd.DataFrame(mpc_rows)
mpc_table.assign(action_mix=mpc_table["action_mix"].astype(str)).to_csv(
    OUT / "B3_mpc.csv", index=False)
print(mpc_table.drop(columns="action_mix").round(4).to_string(index=False))
print()
for row in mpc_rows:
    print(f"  {row['policy']:>10}: {row['action_mix']}")

# Los dos números que la tabla esconde y que hay que decir en voz alta: lo que tarda y lo que
# cuesta en DSCR. Se derivan aquí para no teclearlos a mano en el markdown.
mpc_ms = mpc_table["s_per_company_month"] * 1000
baseline_dscr = baselines.set_index("policy")["dscr_fail_months"]
print(f"\nMPC: {mpc_ms.min():.1f}–{mpc_ms.max():.1f} ms por empresa-mes con {MPC_PATHS} caminos")
print(f"meses con DSCR < {CFG.dscr_floor}: MPC "
      f"{', '.join(f'{k}={int(v)}' for k, v in zip(MPC_K, mpc_table['dscr_fail_months']))} "
      f"frente a do_nothing={int(baseline_dscr['do_nothing'])}, "
      f"advisor_rules={int(baseline_dscr['advisor_rules'])}")
print(f"coste medio en euros, MPC k=0,5 / do_nothing: "
      f"{mpc_table.loc[1, 'mean_cost_eur'] / baselines.loc[0, 'mean_cost_eur']:.1f}×")

**El resultado honesto, con las dos caras:**

| | coste medio (€) | coste medio (meses de cargos) | rotura |
|---|---|---|---|
| no hacer nada | 26.564 | 0,618 | 38,9 % |
| reglas del asesor | 31.740 | 0,580 | 27,8 % |
| MPC k = 0,5 | 794.378 | 0,438 | 9,5 % |

En **euros crudos el MPC es 30 veces más caro** que no hacer nada, y eso hay que decirlo tal
cual: 94,8 M€ de los 100 M€ de coste total son de `COMP_1185`, la empresa de 2.700 M€ de cargos
mensuales, a la que el MPC le va dando préstamos. En **coste relativo el MPC es el más barato de
los tres** (0,438 meses de cargos frente a 0,580 y 0,618) **y** el que menos rompe (9,5 % frente
a 27,8 % y 38,9 %). O sea: con λ ≥ 0,1 × cargos el MPC bate a la regla en la **media**
normalizada (0,421–0,460 frente a 0,580) y en rotura, **empata** con ella en la mediana
normalizada con k = 0,1 y k = 0,5 (0,017) pero la **pierde** con k = 2 (0,030), y **pierde** en
la media en euros del hold-out. Las cifras van juntas o no van: decir solo «MPC mejor en coste y
en rotura» sería falso en la mitad de las columnas.

Cuatro avisos más sobre esta tabla:

- **El MPC compra rotura con cuota, y eso no está en la columna de coste.** Los meses con
  DSCR < 1,2 pasan de **82** (tanto en `do_nothing` como en `advisor_rules`) a **187 / 182 / 162**
  con k = 0,1 / 0,5 / 2: el MPC se lleva la rotura por delante pidiendo préstamos, y el préstamo
  es servicio de deuda todos los meses siguientes. La ficha del asesor tiene que enseñar coste,
  rotura **y** DSCR, o estará vendiendo media película. Nótese además que **más λ baja el DSCR
  fallido** (162 con k = 2 frente a 187 con k = 0,1): con λ alto el MPC prefiere cubrir con la
  póliza, que no genera cuota, antes que endeudarse.
- λ apenas mueve la rotura entre 0,1 y 2 (9,5 % en los tres; el coste relativo sube de 0,421 a
  0,460). No es un error del barrido: el préstamo de la rejilla cuesta ~1,5 % de la mediana de
  cargos en seis meses y λ = 0,1 × cargos ya paga esa prima por 3 puntos de rotura. La frontera
  de verdad aparece en la **mediana** del coste (B5), no en la media.
- `switch_rate` 0,162 / 0,188 / 0,198: el criterio de éxito de §B3 pedía ≤ 20 % de cambios de
  recomendación entre meses consecutivos y se cumple en los tres, con k = 2 por los pelos. La
  regla del asesor cambia el 11,7 %.
- Tiempo: **un par de milisegundos por empresa-mes** con 200 caminos. El valor exacto sale de la
  columna `s_per_company_month` de la tabla y va impreso arriba en milisegundos; oscila entre 2 y
  3 ms según la carga de la máquina. Son tres órdenes de magnitud por debajo del segundo por
  empresa que pedía el criterio, así que el margen no depende de cuál de los dos números se lea.

In [ ]:
def annotate_stacked(ax, xs, ys, labels, digits=(2, 2), dx=7, dy=5, step=11, fontsize=8):
    """Etiqueta puntos apilando los que caen encima (aquí hay políticas que empatan exactas)."""
    seen = Counter()
    for x, y, label in zip(xs, ys, labels):
        key = (round(float(x), digits[0]), round(float(y), digits[1]))
        ax.annotate(label, (x, y), textcoords="offset points",
                    xytext=(dx, dy + step * seen[key]), fontsize=fontsize)
        seen[key] += 1


fig, ax = plt.subplots(figsize=(8, 5))
frame = pd.concat([baselines, mpc_table], ignore_index=True)
for _, row in frame.iterrows():
    marker = "o" if row["policy"].startswith("mpc") else "s"
    ax.scatter(row["breach_rate"], row["mean_cost_rel"], s=70, marker=marker)
annotate_stacked(ax, frame["breach_rate"], frame["mean_cost_rel"], frame["policy"])
ax.set_xlabel("tasa de rotura (alguna vez en 12 meses)")
ax.set_ylabel("coste medio en meses de cargos")
ax.set_title("B2 vs B3: coste normalizado y rotura (126 empresas retenidas, 12 meses)")
ax.margins(x=0.18, y=0.12)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "B3_tradeoff.png", dpi=140)
plt.close(fig)

### Arrepentimiento contra el oráculo

`policies.perfect_foresight` es un **oráculo voraz con previsión realizada a 6 meses**, no el LP
de 12 meses de §B2.v: decide mes a mes viendo los sorteos que van a ocurrir, así que no es cota
superior del valor de cualquier política. Por eso se compara **a `months = cfg.horizon = 6`**,
que es el horizonte que el oráculo optimiza; a 3 meses la desigualdad «oráculo ≤ MPC» puede
fallar legítimamente y no querría decir nada.

El oráculo toma λ y μ como escalares, no como funciones de la empresa. Para mantener el precio
sombra en la escala de cada una, se corre **empresa a empresa** (lista de una sola historia), y
el MPC se corre igual, de modo que los dos ven exactamente el mismo mundo (`[seed, 0, j]`).

In [ ]:
t0 = time.perf_counter()
mpc_single = pl.make_mpc_policy(lam_for(0.5), mu_for(0.5), pool=pool, n_paths=MPC_PATHS)
regret_rows = []
for hist, med_out in zip(hold_hists, hold_med):
    lam, mu = 0.5 * med_out, 0.125 * med_out
    oracle = pl.perfect_foresight([hist], CFG, months=ORACLE_MONTHS, seed=SEED, pool=pool,
                                  lam=lam, mu=mu)
    mpc_one = pl.evaluate_policy(mpc_single, [hist], CFG, months=ORACLE_MONTHS, seed=SEED,
                                 pool=pool)
    objective_mpc = (float(mpc_one["total_cost"].iloc[0])
                     + lam * bool(mpc_one["breach"].iloc[0])
                     + mu * (int(mpc_one["dscr_fail_months"].iloc[0]) > 0))
    regret_rows.append({
        "company_id": hist.company_id, "med_out": med_out,
        "cost_oracle": float(oracle["total_cost"].iloc[0]),
        "cost_mpc": float(mpc_one["total_cost"].iloc[0]),
        "objective_oracle": float(oracle["objective"].iloc[0]),
        "objective_mpc": objective_mpc,
        "breach_oracle": bool(oracle["breach"].iloc[0]),
        "breach_mpc": bool(mpc_one["breach"].iloc[0]),
    })
regret = pd.DataFrame(regret_rows)
regret["regret_eur"] = regret["objective_mpc"] - regret["objective_oracle"]
regret["regret_rel"] = regret["regret_eur"] / np.where(regret["med_out"] > 0,
                                                       regret["med_out"], np.nan)
regret_summary = {
    "months": ORACLE_MONTHS,
    "share_oracle_not_worse": float((regret["regret_eur"] >= -1e-6).mean()),
    "mean_regret_eur": float(regret["regret_eur"].mean()),
    "median_regret_eur": float(regret["regret_eur"].median()),
    "mean_regret_rel": float(np.nanmean(regret["regret_rel"])),
    "cost_oracle_mean_eur": float(regret["cost_oracle"].mean()),
    "cost_mpc_mean_eur": float(regret["cost_mpc"].mean()),
    "breach_oracle": float(regret["breach_oracle"].mean()),
    "breach_mpc": float(regret["breach_mpc"].mean()),
}
print(f"{time.perf_counter() - t0:.1f} s")
for key, value in regret_summary.items():
    print(f"  {key}: {value:,.4f}" if isinstance(value, float) else f"  {key}: {value}")

El oráculo no es peor que el MPC en **el 89,7 % de las empresas**; el 10,3 % restante es ruido
del propio oráculo, que es voraz: elegir lo mejor para el mes en curso viendo seis meses no
garantiza la mejor secuencia. El arrepentimiento mediano es **0 €**: en la mitad de las empresas
el MPC hace exactamente lo que haría alguien que ve el futuro. La media (21 K€) vuelve a ser la
empresa grande.

## B4 — Política aprendida (iteración Q ajustada)

El experimento de RL de §B4, con el método que la propia nota recomienda primero: **FQI con
LightGBM**, tabular, sin GPU. No es el motor de la demo; es la pregunta «¿una política aprendida
en el simulador iguala al MPC con 100× menos cómputo y aguanta mejor un mundo mal
especificado?».

- **Estado** (10 señales, normalizadas por la mediana de cargos para que una pyme de 20 K€ y una
  de 2.700 M€ caigan en el mismo espacio): saldo, bache mínimo, media y desviación de entradas,
  disponible de póliza, cartera, saldo vivo de préstamo, tipo, servicio de deuda y meses de
  historia. **Aviso, y es el que explica el resultado**: esta lista es la del encargo y **no
  incluye los compromisos** que arrastra la `History` (`committed_outflow_m`,
  `committed_cost_m`, `pending_flows`), así que el estado **no es markoviano**: una empresa con
  cero préstamos y la misma con cinco son indistinguibles para Q. Se corre tal cual para
  responder a lo que se preguntó, y se vuelve sobre ello al leer los resultados.
- **Acciones**: la lista fija de siete tipos; cada una se mapea al **primer candidato elegible
  de ese tipo** que devuelve `candidate_actions`, y las no elegibles se enmascaran.
- **Transiciones**: rollouts ε-voraces (ε = 0,3) sobre `advisor_rules` en las **494 empresas de
  train**, 12 meses, 3 repeticiones con semillas distintas → 17.784 transiciones.
- **Recompensa**: `−(coste + λ·rotura_del_mes + μ·fallo_DSCR_del_mes)` con el λ de k = 0,5,
  **dividida por la mediana de cargos de la empresa**. Esta división no estaba en el encargo y es
  una decisión: sin ella una sola Q compartida entre empresas ajusta casi solo a las grandes (el
  error cuadrático de `COMP_1185` es 10⁷ veces el de una pyme) y la política resultante es la de
  la empresa grande aplicada a todas. Se ajustan **las dos variantes** y se reporta la
  diferencia.
- **FQI**: `LGBMRegressor(n_estimators=300, num_leaves=31, learning_rate=0.05)` sobre
  `[estado, acción one-hot]`, K = 8 iteraciones, γ = 0,95, objetivo `r + γ·max_{a' elegible}
  Q(s', a')`. El truncamiento a 12 meses no es terminal: se arranca el valor también en la
  última transición, porque el episodio se corta, no se acaba.

In [ ]:
ACTION_KINDS = ["none", "line_cover", "line_draw", "line_open", "factoring", "loan", "refinance"]
ACTION_INDEX = {kind: i for i, kind in enumerate(ACTION_KINDS)}
STATE_NAMES = ["eom", "min_dip", "mean_inflow", "std_inflow", "line_room", "receivables",
               "loan_outstanding", "loan_rate", "debt_service", "months_history"]
EPSILON, N_REPS, FQI_ITERATIONS, GAMMA, RL_K = 0.3, 3, 8, 0.95, 0.5


def state_vector(hist):
    """Las 10 señales del estado, normalizadas por la mediana de cargos de la empresa."""
    median_out = float(np.median(hist.outflows)) if len(hist.outflows) else 0.0
    unit = median_out if median_out > 0 else 1.0
    rate = hist.loan_rate
    rate = 0.0 if rate is None or not np.isfinite(float(rate)) else float(rate)
    return np.array([
        float(hist.eom) / unit,
        float(np.min(hist.dips)) / unit if len(hist.dips) else 0.0,
        float(np.mean(hist.inflows)) / unit if len(hist.inflows) else 0.0,
        float(np.std(hist.inflows)) / unit if len(hist.inflows) else 0.0,
        (float(hist.line_limit) - float(hist.line_drawn)) / unit,
        float(hist.receivables) / unit,
        float(hist.loan_outstanding) / unit,
        rate,
        float(hist.debt_service_m) / unit,
        float(len(hist.inflows)),
    ])


def action_menu(hist, cfg=CFG):
    """`tipo -> primer candidato elegible de ese tipo`; la máscara de acciones sale de aquí."""
    menu = {}
    for action in pl.candidate_actions(hist, cfg):
        menu.setdefault(action.kind, action)
    return menu


def one_hot(indices):
    matrix = np.zeros((len(indices), len(ACTION_KINDS)))
    matrix[np.arange(len(indices)), np.asarray(indices, dtype=int)] = 1.0
    return matrix


def collect_transitions(hists, cfg=CFG, months=CLOSED_LOOP_MONTHS, epsilon=EPSILON,
                        n_reps=N_REPS, k_lam=RL_K, seed=SEED):
    """Rollouts ε-voraces sobre `advisor_rules`; un paso = `simulate(horizon=1)` + `advance`.

    Reproduce el mundo de `evaluate_policy` (un camino, semilla por empresa-mes) sin tocarlo: la
    semilla lleva la repetición delante para que las tres pasadas vean futuros distintos.
    """
    step_cfg = replace(cfg, n_paths=1)
    states, actions, rewards, next_states, masks, units = [], [], [], [], [], []
    for rep in range(n_reps):
        explorer = np.random.default_rng([seed, 1000 + rep])
        for i, start in enumerate(hists):
            hist = replace(start, inflows=np.array(start.inflows, float),
                           outflows=np.array(start.outflows, float),
                           dips=np.array(start.dips, float))
            median_out = float(np.median(hist.outflows)) if len(hist.outflows) else 0.0
            lam, mu = k_lam * median_out, 0.25 * k_lam * median_out
            unit = median_out if median_out > 0 else 1.0
            for j in range(months):
                menu = action_menu(hist, cfg)
                if explorer.random() < epsilon:
                    kind = ACTION_KINDS[int(explorer.integers(0, len(ACTION_KINDS)))]
                    action = menu.get(kind, pj.NONE)
                else:
                    action = pl.advisor_rules(hist, cfg)
                    if action.kind not in menu:
                        action = pj.NONE
                state = state_vector(hist)
                paths = pj.simulate(hist, action, step_cfg,
                                    rng=np.random.default_rng([seed, rep, i, j]),
                                    pool=pool, horizon=1)
                nxt = pj.advance(hist, paths, action, k=0, cfg=step_cfg)
                reward = -(paths.expected_cost()
                           + lam * (paths.breach_prob() > 0)
                           + mu * (paths.dscr_fail_prob(cfg.dscr_floor) > 0))
                mask = np.zeros(len(ACTION_KINDS))
                for kind in action_menu(nxt, cfg):
                    mask[ACTION_INDEX[kind]] = 1.0
                states.append(state)
                actions.append(ACTION_INDEX[action.kind])
                rewards.append(reward)
                next_states.append(state_vector(nxt))
                masks.append(mask)
                units.append(unit)
                hist = nxt
    return {"state": np.array(states), "action": np.array(actions),
            "reward": np.array(rewards), "next_state": np.array(next_states),
            "mask": np.array(masks), "unit": np.array(units)}


t0 = time.perf_counter()
transitions = collect_transitions(train_hists)
rollout_seconds = time.perf_counter() - t0
taken = Counter(ACTION_KINDS[a] for a in transitions["action"])
print(f"{len(transitions['state']):,} transiciones de {len(train_hists)} empresas × "
      f"{CLOSED_LOOP_MONTHS} meses × {N_REPS} repeticiones en {rollout_seconds:.1f} s")
print(f"acciones tomadas: {dict(taken.most_common())}")

In [ ]:
def fit_q(transitions, normalise=True, iterations=FQI_ITERATIONS, gamma=GAMMA, seed=SEED):
    """Iteración Q ajustada: K regresores LightGBM sobre `[estado, acción one-hot]`."""
    reward = transitions["reward"] / transitions["unit"] if normalise else transitions["reward"]
    X = np.hstack([transitions["state"], one_hot(transitions["action"])])
    next_state = transitions["next_state"]
    tiled = np.repeat(next_state, len(ACTION_KINDS), axis=0)
    tiled_actions = np.tile(np.arange(len(ACTION_KINDS)), len(next_state))
    X_next = np.hstack([tiled, one_hot(tiled_actions)])
    mask = transitions["mask"]
    model, q_next, trace = None, None, []
    for iteration in range(iterations):
        if q_next is None:
            target = reward.copy()
        else:
            values = np.where(mask > 0, q_next.reshape(len(next_state), len(ACTION_KINDS)), -np.inf)
            best = np.where(np.isfinite(values).any(axis=1), values.max(axis=1), 0.0)
            target = reward + gamma * best
        model = LGBMRegressor(n_estimators=300, num_leaves=31, learning_rate=0.05,
                              verbose=-1, random_state=seed)
        model.fit(X, target)
        q_next = model.predict(X_next)
        trace.append({"iteration": iteration + 1, "target_mean": float(target.mean()),
                      "target_std": float(target.std())})
    return model, pd.DataFrame(trace)


def make_q_policy(model, cfg_ref=CFG):
    """`argmax Q` sobre las acciones elegibles; empate a `none`, que va la primera en la lista."""
    def policy(hist, cfg=None, rng=None):
        cfg = cfg or cfg_ref
        menu = action_menu(hist, cfg)
        kinds = [kind for kind in ACTION_KINDS if kind in menu]
        if not kinds:
            return pj.NONE
        features_row = np.hstack([
            np.tile(state_vector(hist), (len(kinds), 1)),
            one_hot([ACTION_INDEX[k] for k in kinds]),
        ])
        return menu[kinds[int(np.argmax(model.predict(features_row)))]]
    return policy


t0 = time.perf_counter()
q_model, q_trace = fit_q(transitions, normalise=True)
q_model_eur, _ = fit_q(transitions, normalise=False)
fqi_seconds = time.perf_counter() - t0
print(f"FQI (dos variantes, K={FQI_ITERATIONS}) en {fqi_seconds:.1f} s")
print(q_trace.round(3).to_string(index=False))

importance = pd.Series(q_model.feature_importances_,
                       index=STATE_NAMES + [f"a_{k}" for k in ACTION_KINDS]).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
importance.plot.barh(ax=ax)
ax.set_xlabel("divisiones usadas por LightGBM")
ax.set_title("Q(s, a): qué mira la política aprendida")
fig.tight_layout()
fig.savefig(OUT / "B4_importance.png", dpi=140)
plt.close(fig)
print(importance.sort_values(ascending=False).head(8).to_string())

La importancia es el proxy de explicabilidad que pedía §B4 (SHAP sobre los Q-valores; aquí, las
divisiones del árbol, que es la versión barata y ordena igual): **el saldo, las entradas medias
y el saldo vivo de préstamo** mandan; el tipo y el bache mínimo casi no se miran. Las columnas
de acción pesan poquísimo, que es la primera señal de alarma: el valor lo pone el estado, no la
acción, y una Q así recomienda casi lo mismo pase lo que pase.

In [ ]:
rl_rows = []
shift_specs = {
    "advisor_rules": pl.advisor_rules,
    "mpc_k0.5": pl.make_mpc_policy(lam_for(0.5), mu_for(0.5), pool=pool, n_paths=MPC_PATHS),
    "fqi": make_q_policy(q_model),
    "fqi_eur": make_q_policy(q_model_eur),
}
for name, policy in shift_specs.items():
    for world, shift in (("base", None), ("shift", SHIFT)):
        t0 = time.perf_counter()
        run = pl.evaluate_policy(policy, hold_hists, CFG, months=CLOSED_LOOP_MONTHS, seed=SEED,
                                 pool=pool, shift=shift)
        row = summarise(f"{name}/{world}", run, time.perf_counter() - t0, len(hold_hists))
        row["world"] = world
        row["base_policy"] = name
        rl_rows.append(row)

rl_table = pd.DataFrame(rl_rows)
rl_table.assign(action_mix=rl_table["action_mix"].astype(str)).to_csv(OUT / "B4_rl.csv",
                                                                     index=False)
print(rl_table[["policy", "mean_cost_eur", "median_cost_eur", "mean_cost_rel", "breach_rate",
                "dscr_fail_months", "share_none", "s_per_company_month"]].round(4).to_string(
                    index=False))
print()
for row in rl_rows:
    print(f"  {row['policy']:>18}: {row['action_mix']}")

**El FQI no se gana el sitio, y el motivo es instructivo.** El criterio de §B4 era «≥ MPC − 5 %
en coste y ≤ +1 punto de rotura, o mejor que MPC bajo desplazamiento», y no cumple ninguna de las
dos ramas. En el mundo base pide **préstamo el 78 % de los meses** (1.187 de 1.512), sube el
coste relativo un 60 % sobre el MPC (0,700 frente a 0,438 meses de cargos — peor incluso que no
hacer nada, 0,618), dobla los meses con DSCR bajo el suelo frente al MPC (407 frente a 182) y
los quintuplica frente a las reglas (82). A cambio rompe un 12,7 % frente al 9,5 % del MPC: ni
siquiera compra la rotura que paga.

**La causa principal es que el estado no es markoviano.** El vector de estado que fija el
encargo tiene diez señales y **ninguna es un compromiso**: `committed_outflow_m`,
`committed_cost_m` y `pending_flows` —justo lo que `advance` escribe cuando la acción anterior
deja algo pagando— se quedan fuera. Para Q, una empresa que no ha pedido ningún préstamo y la
misma empresa después de pedir cinco son **el mismo estado** si el saldo y los flujos coinciden,
de modo que el regresor no puede aprender que el sexto préstamo es peor que el primero. Con esa
amputación, la iteración Q está ajustando el valor de una acción sobre un estado que no
determina su consecuencia, y la política que sale de ahí no puede ser buena. Arreglarlo es
barato —tres señales más, normalizadas por la mediana de cargos— y es el primer experimento que
habría que correr antes de volver a juzgar al FQI.

**La segunda causa es un agujero del simulador que la política encuentra sola**, que es el
riesgo que documenta §3 (*la política explota los errores del modelo*): en `projection._plan`
**el primer mes del préstamo es de carencia**. Con un paso de un mes, pedir un préstamo entra
caja y no cuesta nada *ese mes*; la cuota y el interés los escribe `advance` para los meses que
vienen —y, como acabamos de decir, el estado no los ve—. Un FQI con γ = 0,95 y 12 pasos debería
descontar la consecuencia a través del arranque del valor, pero los objetivos divergen a lo
largo de las ocho iteraciones (la desviación del objetivo pasa de 27 a 93), que es la
inestabilidad clásica del bootstrapping con aproximador no lineal fuera del soporte de los
datos. El MPC no cae en la trampa porque **simula los seis meses enteros antes de decidir**, con
los compromisos dentro de la `History`.

Sobre las dos variantes de recompensa: en el mundo base acaban en el mismo coste relativo
(0,700 y 0,695) y la de euros rompe algo más (17,5 % frente a 12,7 %), o sea que ahí el problema
no es la escala. **Bajo desplazamiento sí se separan del todo**: la normalizada rompe el 19,0 %
y sigue pidiendo préstamos (`none` el 19 % de los meses), mientras que la de euros crudos se
paraliza (`none` el 74 %) y se va al 62,7 % de rotura, peor que las reglas. La variante en euros
es una política de la empresa grande aplicada a todas, y en cuanto el mundo se mueve deja de
valer: normalizar la recompensa no arregla el FQI, pero no hacerlo lo empeora.

Bajo desplazamiento (entradas −20 %, baches +30 %, tipos +200 pb) degradan las tres políticas; el
MPC es el que mejor aguanta en coste relativo (0,958 frente a 1,121 de las reglas y 1,243 del
FQI). El único punto a favor del FQI está aquí: rompe menos que el MPC (19,0 % frente a 25,4 %)
porque su manía de pedir préstamos resulta ser un colchón; pero cuesta un 30 % más, así que
tampoco lo domina.

Conclusión para el pitch, literal: **el motor es MPC; el RL se probó, explotó un agujero del
simulador y se queda como transparencia**. Para que tuviera sentido haría falta lo que dice la
propia nota: penalizar el desacuerdo de un ensemble de simuladores (MOPO) o usar la Q como
aproximación del valor terminal *dentro* del MPC, no en su lugar.

## B5 — Frontera coste–riesgo

Barrido de `k ∈ {0,1; 0,25; 0,5; 1; 2; 4}` sobre una submuestra de 100 empresas retenidas, 6
meses y 100 caminos por decisión, que es donde la frontera se ve sin que la empresa grande se
coma la escala. Es la tabla que el asesor lee como «conservador / equilibrado / barato».

In [ ]:
subsample_idx = np.sort(np.random.default_rng(SEED).choice(
    len(hold_hists), size=min(FRONTIER_N, len(hold_hists)), replace=False))
subsample = [hold_hists[i] for i in subsample_idx]
subsample_unit = np.where(hold_med[subsample_idx] > 0, hold_med[subsample_idx], np.nan)

frontier_rows = []
for k in FRONTIER_K:
    policy = pl.make_mpc_policy(lam_for(k), mu_for(k), pool=pool, n_paths=FRONTIER_PATHS)
    t0 = time.perf_counter()
    run = pl.evaluate_policy(policy, subsample, CFG, months=ORACLE_MONTHS, seed=SEED, pool=pool)
    cost = run["total_cost"].to_numpy(float)
    relative = cost / subsample_unit
    kinds = Counter(kind for actions in run["actions"] for kind in actions)
    frontier_rows.append({
        "k": k,
        "mean_cost_eur": float(cost.mean()),
        "median_cost_eur": float(np.median(cost)),
        "mean_cost_rel": float(np.nanmean(relative)),
        "median_cost_rel": float(np.nanmedian(relative)),
        "breach_rate": float(run["breach"].mean()),
        "breach_months": int(run["n_breach_months"].sum()),
        "dscr_fail_months": int(run["dscr_fail_months"].sum()),
        "share_none": kinds.get("none", 0) / sum(kinds.values()),
        "seconds": time.perf_counter() - t0,
    })

frontier = pd.DataFrame(frontier_rows)
frontier.to_csv(OUT / "B5_frontier.csv", index=False)
print(frontier.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
ax = axes[0]
ax.scatter(frontier["breach_rate"], frontier["median_cost_eur"], s=70)
annotate_stacked(ax, frontier["breach_rate"], frontier["median_cost_eur"],
                 [f"k={k:g}" for k in frontier["k"]], digits=(3, 0))
ax.set_xlabel("tasa de rotura a 6 meses")
ax.set_ylabel("coste mediano (€)")
ax.set_title("Frontera coste–riesgo: los seis λ se apilan")
ax.margins(x=0.25, y=0.18)
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(frontier["k"], frontier["median_cost_eur"], "o-", color="tab:blue",
        label="coste mediano (€)")
ax.set_xscale("log")
ax.set_xticks(list(FRONTIER_K), [f"{k:g}" for k in FRONTIER_K])
ax.set_xlabel("k  (λ = k × mediana de cargos)")
ax.set_ylabel("coste mediano (€)", color="tab:blue")
twin = ax.twinx()
twin.plot(frontier["k"], frontier["breach_rate"], "s--", color="tab:red",
          label="tasa de rotura")
twin.set_ylim(0, max(0.05, frontier["breach_rate"].max() * 1.6))
twin.set_ylabel("tasa de rotura a 6 meses", color="tab:red")
ax.set_title("λ compra coste, no riesgo")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "B5_frontier.png", dpi=140)
plt.close(fig)

La frontera **existe pero es casi plana**: la rotura se queda en 2–3 % para todo `k`, y lo que
se mueve monótonamente es el coste —la mediana de 484 € a 1.127 € y la media normalizada de
0,105 a 0,110 meses de cargos—, no el riesgo. Traducido: en este dataset
el MPC ya compra casi toda la protección disponible con el λ más barato, y subir el precio
sombra solo hace que pague de más. Para la pantalla del asesor eso significa que los tres
botones «conservador / equilibrado / barato» hay que ponerlos en `k ∈ {0,1; 0,5; 2}` sabiendo
que la diferencia que el usuario va a ver es **de coste, no de riesgo**; venderla como una
elección de riesgo sería vender humo.

## Cierre

Lo que se lleva la pista B, en orden de lo que hay que decir en el pitch:

1. **El simulador ordena (AUC 0,79 a un mes; 0,69 a seis) y no calibra sin la isotónica.** Se
   enseña la probabilidad calibrada, siempre.
2. **Como clasificador empata con el score de reglas** (0,698 frente a 0,698). Su valor es el
   contrafactual —«¿y si abro una póliza de 200 K€?»— y los euros, no el ranking.
3. **La mecánica de los productos es compatible con los ATT observados** en dos de tres casos; en
   el préstamo el signo se invierte porque A2 mide selección y B1 mide mecánica.
4. **El MPC baja la rotura del 27,8 % (reglas) al 9,5 %** y es el más barato en media
   normalizada (0,438 frente a 0,580 meses de cargos), empata en la mediana con k ≤ 0,5 y **en
   euros brutos es más caro** porque una empresa se lleva el 95 % del coste. Y **lo paga en
   DSCR**: 162–187 meses bajo el suelo frente a 82 de las reglas y de no hacer nada, porque
   compra la rotura con préstamos. Las cuatro cifras van juntas o no van; la ficha del asesor
   tiene que enseñar coste, rotura y DSCR.
5. **El arrepentimiento mediano contra el oráculo a 6 meses es 0 €**, y el oráculo no es peor en
   el 89,7 % de las empresas.
6. **El FQI falla y se cuenta**: el estado del encargo no lleva los compromisos, así que no es
   markoviano y Q no puede distinguir el primer préstamo del sexto; encima el mes de carencia
   hace que a un paso el préstamo parezca gratis. Bajo desplazamiento rompe menos que el MPC
   (19,0 % frente a 25,4 %) pero cuesta un 30 % más. El motor se queda en MPC.
7. **λ mueve el coste, no el riesgo**: la frontera es plana en rotura entre k = 0,1 y k = 4.

Y las limitaciones, todas medidas, ninguna estimada: banda al 72 % en vez del 80 %; compromisos
sin plazo en `advance`; interés del préstamo comprometido plano; oráculo voraz, no LP; ADL sin
una sola empresa elegible; 126 empresas retenidas en vez de 150.

In [ ]:
summary = {
    "generated_at": pd.Timestamp.now("UTC").isoformat(),
    "seed": SEED,
    "config": {
        "n_paths_card": CFG.n_paths, "n_paths_backtest": pj.BACKTEST_PATHS,
        "n_paths_mpc": MPC_PATHS, "horizon": CFG.horizon,
        "closed_loop_months": CLOSED_LOOP_MONTHS, "oracle_months": ORACLE_MONTHS,
        "backtest_months": BACKTEST_MONTHS, "train_until": TRAIN_UNTIL,
        "start_month": START_MONTH, "shift": SHIFT,
    },
    "split": {
        "n_groups": int(group_of.nunique()),
        "hold_out_companies": len(hold_ids), "hold_out_qualifying": len(hold_hists),
        "train_companies": len(train_ids), "train_qualifying": len(train_hists),
        "eligible_line_room": n_line_room, "eligible_refinance": n_refi,
        "eligible_receivables": n_receivables,
        "median_outflow_eur": {"p25": float(np.percentile(hold_med, 25)),
                               "median": float(np.median(hold_med)),
                               "p75": float(np.percentile(hold_med, 75)),
                               "max": float(hold_med.max()),
                               "zeros": int((hold_med == 0).sum())},
    },
    "B1_backtest": {
        "with_pool": {k: v for k, v in metrics_pool.items() if k != "calibration"},
        "without_pool": {k: v for k, v in metrics_nopool.items() if k != "calibration"},
        "calibration_deciles": metrics_pool["calibration"],
        "auc_same_rows": {"n_rows": int(ok.sum()), **{k: float(v) for k, v in auc_rows.items()}},
        "validation_mechanics": validation.round(6).to_dict(orient="records"),
    },
    "B2_baselines": baselines.assign(action_mix=baselines["action_mix"].astype(str)).to_dict(
        orient="records"),
    "B3_mpc": mpc_table.assign(action_mix=mpc_table["action_mix"].astype(str)).to_dict(
        orient="records"),
    "B3_regret_vs_oracle": regret_summary,
    "B4_rl": rl_table.assign(action_mix=rl_table["action_mix"].astype(str)).to_dict(
        orient="records"),
    "B4_fqi": {
        "n_transitions": int(len(transitions["state"])),
        "epsilon": EPSILON, "n_reps": N_REPS, "iterations": FQI_ITERATIONS, "gamma": GAMMA,
        "lambda_k": RL_K, "rollout_seconds": rollout_seconds, "fqi_seconds": fqi_seconds,
        "target_trace": q_trace.to_dict(orient="records"),
        "importance": importance.sort_values(ascending=False).astype(int).to_dict(),
    },
    "B5_frontier": frontier.to_dict(orient="records"),
    "caveats": [
        "Los compromisos de `advance` no llevan plazo: la cuota de un préstamo se sigue cobrando "
        "más allá del vencimiento en rollouts largos (irrelevante a 6-12 meses, no a 24).",
        "El interés del préstamo se compromete plano, al tipo del primer mes, porque un float no "
        "lleva cuadro de amortización; el error va del lado caro.",
        "La banda 10-90 % cubre ~72 %, no 80 %: los sorteos son i.i.d. y no hay persistencia de "
        "régimen.",
        "La P(rotura) cruda sobreestima (decil alto: 0,94 predicho frente a 0,23 realizado); se "
        "enseña siempre la calibrada con la isotónica de entrenamiento.",
        "`perfect_foresight` es un oráculo VORAZ con previsión realizada a 6 meses, no el LP de 12 "
        "meses de §B2.v: no es cota superior y solo se compara a months = cfg.horizon = 6.",
        "El coste medio en euros está dominado por COMP_1185 (2.700 M€ de cargos mensuales, 95 % "
        "del coste total del MPC); el coste relativo a la mediana de cargos es el comparable.",
        "adl_refinance no tiene ni una empresa elegible en el hold-out (debt_schedule_config solo "
        "cubre 87 contratos), así que combine(adl, rules) sale idéntico a rules: no es un empate.",
        "El hold-out son 126 empresas, no las 150 del encargo; relajar a 6 meses de futuro sube a "
        "130 y tampoco llega, así que se mantienen el criterio estricto y los 12 meses de bucle.",
        "La recompensa del FQI se divide por la mediana de cargos de la empresa (no estaba en el "
        "encargo); se ajusta también la variante en euros. En el mundo base las dos acaban igual "
        "en coste relativo (0,700 y 0,695), pero bajo desplazamiento se separan: la de euros se "
        "paraliza (none 74 % de los meses) y se va al 62,7 % de rotura frente al 19,0 %.",
        "El estado del FQI que fija el encargo NO incluye los compromisos "
        "(`committed_outflow_m`, `committed_cost_m`, `pending_flows`), así que no es markoviano: "
        "una empresa con cero préstamos y la misma con cinco son el mismo estado para Q, que por "
        "tanto no puede aprender que el sexto préstamo es peor que el primero. Es la causa "
        "principal del fallo de B4 y el primer arreglo que habría que probar.",
        "Además, el FQI explota la carencia del primer mes del préstamo, que con paso de un mes "
        "sale gratis; los objetivos de FQI divergen a lo largo de las 8 iteraciones.",
        "MPC no es más barato que las reglas en la media en euros; lo es en coste normalizado y "
        "en rotura. Las dos cifras se dan juntas.",
        "El MPC compra rotura con cuota: dobla los meses con DSCR bajo el suelo (187/182/162 con "
        "k = 0,1/0,5/2) frente a los 82 de do_nothing y advisor_rules. La ficha del asesor tiene "
        "que enseñar coste, rotura Y DSCR.",
        "B1.iii simula en el mes ANTERIOR al evento: una History es el estado al cierre de su "
        "mes, así que la del mes del evento ya lleva dentro el producto adoptado.",
        "El préstamo de B1.iii se dimensiona invirtiendo la anualidad a 36 meses (el plazo de "
        "SimConfig), no con el `24 × cuota` del encargo, que daba un principal 0,70× del que "
        "reproduce la cuota observada.",
    ],
    "wall_clock_seconds": round(time.time() - T0, 1),
}
with (OUT / "B_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, ensure_ascii=False, indent=2, default=float)
print(json.dumps(summary, ensure_ascii=False, indent=2, default=float))